# ÁP BỘ PHÂN XỬ CẶP LÊN ĐỀ THI — dựng bài nộp

Bộ phân xử (`judge_pair`) đã đạt ngưỡng trên dev300: **53/65 = 81,5%** trên dải
margin < 0,005, hoà vốn 70,8%, **NET +7 câu = +2,33 điểm**.

## Cấu hình đúng như lúc ĐO — không được đổi
| | |
|---|---|
| bộ điểm nền | `scores_public_M20K50_90e09359.json` (md5 `90e09359`) |
| xếp hạng | `max(ce, ce_deep)` |
| dải can thiệp | **`|margin| < 0,005`** — margin = điểm hạng 1 − điểm hạng 2 |
| đoạn đưa cho phân xử | `pick_chunks(k=1)` cho mỗi văn bản — khớp lúc huấn luyện |
| quyết định | chấm **cả hai chiều** (A,B) và (B,A), lấy trung bình, > 0 thì giữ hạng 1 |

⚠️ **Áp lên bộ chấm GỐC (bài 0,702), KHÔNG áp lên bài FT 0,711.** Đó là cấu hình đã đo.
Ghép cả hai là tổ hợp **chưa đo** và hai thứ sửa cùng một quyết định (quy tắc 7).
Ô 3 in số chồng lấn với bài FT để biết, miễn phí.

## Dự kiến
**0,702 + 2,33 ≈ 0,725** · trên bài đang nộp 0,711 · trên sàn nhiễu LB 1,45 nên **đo được**.

## NGƯỠNG NỘP — khoá trước
> LB **> 0,711** ⇒ giữ. **≤ 0,711** ⇒ nộp lại `sub_FT_N3_k50.zip` **NGAY TRONG NGÀY**.
> Bảng hiển thị bài MỚI NHẤT, không phải bài tốt nhất.

## Cổng kiểm (bẫy ⑧ + quy tắc 9)
Ô 2 phải dựng lại được bài 0,702 khớp **1000/1000** TRƯỚC khi đọc bất kỳ số nào.
Nếu số câu đổi = 0 thì dải rỗng — dừng, đừng nộp.


In [ ]:
import os, sys, json, time, zipfile, hashlib
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

assert torch.cuda.is_available(), "CHUA BAT GPU -> Settings > Accelerator > GPU T4"
print(f"  GPU: {torch.cuda.get_device_name(0)}")

ROOT = "/kaggle/input"
def find(name):
    for r, _, fs in os.walk(ROOT):
        if name in fs: return os.path.join(r, name)
    raise FileNotFoundError(f"KHONG THAY {name} duoi {ROOT}")
def find_judge():
    for r, ds, fs in os.walk(ROOT):
        if os.path.basename(r) == "judge_pair" and "config.json" in fs: return r
    raise FileNotFoundError("KHONG THAY thu muc judge_pair")

# ===== VAN TAY BO DIEM (quy tac 8: nhan dien bang RUOT, khong bang TEN) =====
SIG_MD5, SIG_NCAND, SIG_NDEEP = "90e0935918d7ef96de04f9acf4a9ec6b", 50, 20000
SCORESF = find("scores_public_M20K50_90e09359.json")
_h = hashlib.md5(open(SCORESF, "rb").read()).hexdigest()
print(f"  bo diem md5={_h}")
assert _h == SIG_MD5, f"SAI BO DIEM\n  co  {_h}\n  can {SIG_MD5}"

TESTF = find("public-official.json")
JUDGE = find_judge()
UTIL  = os.path.dirname(find("deep_chunk.py")); sys.path.insert(0, UTIL)
import deep_chunk as DC
DC.MERGE_CHARS = 1800
CTX = None
for _r, _, _fs in os.walk(ROOT):
    if any(f.startswith("context_") and f.endswith(".json") for f in _fs): CTX = _r; break
assert CTX and len(DC.read_passage(CTX, os.listdir(CTX)[0][8:-5])) > 50, "CTX sai"
for n, v in [("SCORES", SCORESF), ("TEST", TESTF), ("JUDGE", JUDGE), ("CTX", CTX)]:
    print(f"  {n:<7} {v}")

test = json.load(open(TESTF, encoding="utf-8-sig"))
S    = json.load(open(SCORESF, encoding="utf-8"))
Q    = list(test)
assert set(S) == set(Q), "qid khong khop de thi"
_nc = {len(S[q]) for q in Q}; _nd = sum(1 for q in Q for v in S[q].values() if "ce_deep" in v)
print(f"  ung vien/cau={sorted(_nc)} · ban ghi co ce_deep={_nd:,}")
assert _nc == {SIG_NCAND} and _nd >= SIG_NDEEP, "hinh dang bo diem sai"

mx    = lambda v: max(v["ce"], v.get("ce_deep", -9e9))
order = {q: [d for d, _ in sorted(S[q].items(), key=lambda kv: -mx(kv[1]))] for q in Q}
CUR   = {q: order[q][0] for q in Q}                   # = dung bai 0.702
qtext = lambda q: test[q]["question"] if isinstance(test[q], dict) else test[q]
margin= {q: mx(S[q][order[q][0]]) - mx(S[q][order[q][1]]) for q in Q}

BAND = 0.005                                          # DA KHOA — dung doi
band = [q for q in Q if margin[q] < BAND]
print(f"\n  dung lai bai 0.702 tu bo diem: {len(CUR)} cau")
print(f"  dai margin < {BAND}: {len(band)}/{len(Q)} cau = {len(band)/len(Q):.1%}  <- so cau se can thiep")
assert len(band) > 0, "dai rong -> khong co gi de lam"

MAXLEN = 1536
tok = AutoTokenizer.from_pretrained(JUDGE)
model = AutoModelForSequenceClassification.from_pretrained(JUDGE).cuda().eval()
print(f"  nap bo phan xu: {sum(p.numel() for p in model.parameters())/1e9:.3f}B")


In [ ]:
# ===== Cham bo phan xu tren dai margin nho — CA HAI CHIEU, luu rieng tung chieu =====
# Luu s1 (A,B) va s2 (B,A) RIENG. Chi cau co DU ca hai chieu moi duoc coi la xong
# -> chay lai la noi tiep an toan, khong the lay trung binh cua mot chieu.
OUT = "/kaggle/working/judge_public.json"
st  = json.load(open(OUT, encoding="utf-8")) if os.path.isfile(OUT) else {"s1": {}, "s2": {}, "skip": {}}
xong = lambda: {q for q in st["s1"] if q in st["s2"]} | set(st["skip"])
print(f"da co {len(xong())}/{len(band)} cau")

def enc(qs, As, Bs):
    ts = [f"[A] {a} [B] {b}" for a, b in zip(As, Bs)]
    return tok(qs, ts, truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")

def cham(qs, As, Bs):
    e = enc(qs, As, Bs)
    with torch.autocast("cuda", dtype=torch.float16):
        return model(**{k: v.cuda() for k, v in e.items()}).logits.view(-1).float().cpu().tolist()

t0 = time.time(); B = 8
todo = [q for q in band if q not in xong()]
with torch.no_grad():
    for i in range(0, len(todo), B):
        qs, As, Bs, ok = [], [], [], []
        for q in todo[i:i+B]:
            d1, d2 = order[q][:2]; qt = qtext(q)
            try:
                c1 = (DC.pick_chunks(qt, CTX, d1, k=1) or [""])[0][:1800]
                c2 = (DC.pick_chunks(qt, CTX, d2, k=1) or [""])[0][:1800]
            except Exception as ex:
                st["skip"][q] = str(ex)[:80]; continue      # khong doc duoc -> GIU hang 1
            if not c1.strip() or not c2.strip():
                st["skip"][q] = "doan rong"; continue
            qs.append(qt); As.append(c1); Bs.append(c2); ok.append(q)
        if ok:
            for q, v in zip(ok, cham(qs, As, Bs)): st["s1"][q] = v      # chieu (A,B)
            for q, v in zip(ok, cham(qs, Bs, As)): st["s2"][q] = v      # chieu (B,A)
        json.dump(st, open(OUT, "w"))
        if (i // B) % 6 == 0:
            print(f"  {len(xong())}/{len(band)} · {(time.time()-t0)/60:.1f} phut", flush=True)

json.dump(st, open(OUT, "w"))
print(f"XONG · {len(xong())}/{len(band)} cau · {(time.time()-t0)/60:.1f} phut")
print(f"  bo qua (khong doc duoc doan): {len(st['skip'])} cau -> giu nguyen hang 1")

# diem cuoi = trung binh hai chieu. >0 = giu hang 1 · <=0 = chon hang 2
res = {q: (st["s1"][q] - st["s2"][q]) / 2 for q in st["s1"] if q in st["s2"]}
for q in st["skip"]: res[q] = 1.0
assert len(res) == len(band), f"{len(res)} != {len(band)}"
_n2 = sum(1 for v in res.values() if v <= 0)
print(f"  phan xu chon hang 2 o {_n2}/{len(band)} cau = {_n2/len(band):.1%}")


In [ ]:
# ===== Dung bai nop =====
assert len(res) == len(band), f"moi cham {len(res)}/{len(band)}"

pick = dict(CUR)
doi  = []
for q in band:
    if res[q] <= 0:                      # phan xu chon HANG 2
        pick[q] = order[q][1]; doi.append(q)

print(f"bo phan xu doi {len(doi)}/{len(band)} cau trong dai · {len(doi)}/{len(Q)} tren ca bo")
assert len(doi) > 0, "phan xu khong doi cau nao -> khong co gi de nop"

# --- CUA KIEM: moi cau van dung 1 id, va chi doi trong dai ---
assert all(pick[q] == CUR[q] for q in Q if q not in band), "doi cau NGOAI dai -> sai logic"
sub = {q: {"answer": [pick[q]]} for q in Q}
assert len(sub) == 1000 and set(sub) == set(Q)
assert all(len(v["answer"]) == 1 and isinstance(v["answer"][0], str) for v in sub.values())

Z = "/kaggle/working/sub_JUDGE_band005.zip"
with zipfile.ZipFile(Z, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("submission.json", json.dumps(sub, ensure_ascii=False))
print(f"da dung {Z}")

# --- CHONG LAP voi bai FT 0,711 (mien phi, chi de BIET — khong dung de quyet) ---
try:
    zf = zipfile.ZipFile(find("sub_FT_N3_k50.zip"))
    FT = {q: v["answer"][0] for q, v in json.loads(zf.read("submission.json")).items()}
    ft_doi = {q for q in Q if FT[q] != CUR[q]}
    print(f"\n[chi de biet] bai FT doi {len(ft_doi)} cau · phan xu doi {len(doi)} cau")
    print(f"  chong lap: {len(ft_doi & set(doi))} cau")
    print(f"  => hai huong {'CHONG LAP NHIEU' if len(ft_doi & set(doi)) > len(doi)*0.3 else 'gan nhu ROI NHAU'}")
    print("  (quy tac 7: KHONG cong don truoc khi do that. Bai nay chi co phan xu.)")
except Exception as e:
    print(f"\n[chi de biet] khong doi chieu duoc voi bai FT: {e}")

print(f"""
====================================================================
NOP: sub_JUDGE_band005.zip
  du kien  : 0,702 + 2,33 = ~0,725   (bai dang nop 0,711 · san nhieu LB 1,45)
  NGUONG   : LB > 0,711 -> GIU
             LB <= 0,711 -> NOP LAI sub_FT_N3_k50.zip NGAY TRONG NGAY
  Bang hien thi bai MOI NHAT, khong phai bai tot nhat.
====================================================================""")
